# Pre-encoded BEHAVIOR Dataset Tests

Validates that `quastAI/behavior2025-vjepa2-vitg-130h-demo-embeddings` (MDS shards) was built
correctly from the base dataset.
**What these tests cannot guarantee:**
- Visual quality of decoded video frames (pixel correctness is not checked)
- Whether V-JEPA2 produced semantically meaningful embeddings
- Correctness for episodes not sampled in the spot-checks (Tests 4 & 5)

In [1]:
# pip install huggingface_hub mosaicml-streaming pandas numpy pyarrow opencv-python

import json
import os
from collections import defaultdict
from math import ceil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from streaming.base.format.mds.reader import MDSReader

# ── Config — must match configs/behavior-vitg16-256px-16f.yaml ───────────────
HF_REPO_ID    = "quastAI/behavior2025-vjepa2-vitg-130h-demo-embeddings"
HF_SUBFOLDER  = "shards_256px_vit_16_g"
BASE_DATASET  = Path("/Users/julianquast/Documents/Documents/LOTO-H-JEPA/dataset_pipeline/base_dataset")
MANIFEST_PATH = BASE_DATASET / "manifest.json"

TARGET_FPS  = 5
FPC         = 8
STATE_DIM   = 7
STATE_START = 0
ACTION_DIM  = 23
NATIVE_FPS  = 30
FSTP_NOM    = ceil(NATIVE_FPS / TARGET_FPS)  # = 6

N_SPOT_CHECK = 5   # episodes cross-checked against raw parquet

print(f"FSTP_NOMINAL={FSTP_NOM}  (expected step between sampled frames)")

/Users/julianquast/Documents/Documents/LOTO-H-JEPA/dataset_pipeline/BEHAVIOR_challenge_dataset/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/Cellar/python@3.10/3.10.20_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, Non

FSTP_NOMINAL=6  (expected step between sampled frames)


---
## Test 1 — Download & Schema Validation

**Guarantees:** The HF repo can be downloaded; `index.json` is valid; every shard contains all
expected MDS columns; the dataset is non-empty; first-row array shapes match pipeline config.

**Does NOT guarantee:** Numerical correctness of any values.

In [ ]:

import os
os.environ["HF_TOKEN"] = ""
print("Downloading encoded shards from Hugging Face …")
local_repo = Path(
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        allow_patterns=[f"{HF_SUBFOLDER}/**"],
    )
)
encoded_dir = local_repo / HF_SUBFOLDER
print(f"Local path: {encoded_dir}")
assert encoded_dir.exists(), f"Expected directory not found: {encoded_dir}"

index_path = encoded_dir / "index.json"
assert index_path.exists(), "index.json missing — dataset may be incomplete"
with open(index_path) as f:
    index = json.load(f)

shards = index.get("shards", [])
assert len(shards) > 0, "index.json contains no shards"
total_rows = sum(s["samples"] for s in shards)
print(f"Shards: {len(shards)},  total rows (frames): {total_rows:,}")
assert total_rows > 0, "Dataset is empty"

EXPECTED_COLUMNS = {"tokens", "actions", "states", "frame_index",
                    "episode_idx", "sample_idx", "frame_pos", "episode_len"}
for shard_info in shards:
    cols = {c["name"] for c in shard_info.get("column_encodings", [])}
    missing = EXPECTED_COLUMNS - cols
    assert not missing, f"Shard {shard_info['raw_data']['basename']} missing columns: {missing}"

print("Schema OK.  Loading all rows into memory …")
reader = MDSReader(out=str(encoded_dir))
rows = [reader[i] for i in range(len(reader))]
print(f"Loaded {len(rows):,} rows.")

r0 = rows[0]
assert isinstance(r0["tokens"],  np.ndarray) and r0["tokens"].ndim  == 2
assert isinstance(r0["actions"], np.ndarray) and r0["actions"].ndim == 1
assert isinstance(r0["states"],  np.ndarray) and r0["states"].ndim  == 1
assert r0["actions"].shape[0] == FSTP_NOM * ACTION_DIM, \
    f"actions: expected {FSTP_NOM * ACTION_DIM}, got {r0['actions'].shape[0]}"
assert r0["states"].shape[0] == STATE_DIM, \
    f"states: expected {STATE_DIM}, got {r0['states'].shape[0]}"

TOKEN_SHAPE = r0["tokens"].shape
print(f"Token shape per frame: {TOKEN_SHAPE}  "
      f"(tokens_per_frame={TOKEN_SHAPE[0]}, hidden_dim={TOKEN_SHAPE[1]})")
print("PASS Test 1")

Fetching 33 files:   3%|▎         | 1/33 [01:29<47:39, 89.37s/it]


KeyboardInterrupt: 

---
## Test 2 — Episode Completeness

**Guarantees:** For every episode: `frame_pos` is exactly `0 … episode_len-1` with no gaps and
no duplicates, and `episode_len` is consistent across all rows of that episode.  Rules out lost
or duplicated frames during the episode-accumulation / flush phase in `BehaviorEpisodePreencoder`.

**Does NOT guarantee:** That `episode_len` equals the expected number of sampled frames from the
source video (see Test 3 for that).

In [ ]:
episodes: dict = defaultdict(list)
for row in rows:
    episodes[row["episode_idx"]].append(row)

print(f"Unique episodes in MDS: {len(episodes)}")

failures = []
for ep_id, ep_rows in episodes.items():
    stored_len = ep_rows[0]["episode_len"]
    if not all(r["episode_len"] == stored_len for r in ep_rows):
        failures.append(f"ep {ep_id}: inconsistent episode_len values")
        continue
    if len(ep_rows) != stored_len:
        failures.append(f"ep {ep_id}: stored episode_len={stored_len} but found {len(ep_rows)} rows")
        continue
    positions = sorted(r["frame_pos"] for r in ep_rows)
    if positions != list(range(stored_len)):
        failures.append(f"ep {ep_id}: frame_pos is not 0..{stored_len-1}")

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} episode completeness failures")

print(f"PASS Test 2 — all {len(episodes)} episodes have contiguous frame_pos with correct episode_len.")

---
## Test 3 — Frame-Index Sampling Pattern

**Guarantees:** For every episode, stored `frame_index` values are strictly monotonically increasing
and the step between consecutive frames equals `ceil(video_fps / TARGET_FPS)`.  Confirms the video
was subsampled at exactly `TARGET_FPS=5`, not densely or randomly.

**Does NOT guarantee:** That the decoded pixel content of those frames is correct.

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

def video_path(sample_idx):
    ep = manifest["episodes"][sample_idx]
    name = os.path.splitext(os.path.basename(ep["episode_file"]))[0]
    return BASE_DATASET / ep["task_name"] / "video" / f"{name}.mp4"

def parquet_path(sample_idx):
    ep = manifest["episodes"][sample_idx]
    name = os.path.splitext(os.path.basename(ep["episode_file"]))[0]
    return BASE_DATASET / ep["task_name"] / "data" / f"{name}.parquet"

failures = []
for ep_id, ep_rows in episodes.items():
    sorted_rows = sorted(ep_rows, key=lambda r: r["frame_pos"])
    fis = [r["frame_index"] for r in sorted_rows]

    if any(fis[i] >= fis[i+1] for i in range(len(fis)-1)):
        failures.append(f"ep {ep_id}: frame_index not strictly increasing")
        continue

    vp = video_path(sorted_rows[0]["sample_idx"])
    if not vp.exists():
        print(f"  SKIP ep {ep_id}: video not found locally")
        continue
    cap = cv2.VideoCapture(str(vp))
    fstp = max(1, ceil(cap.get(cv2.CAP_PROP_FPS) / TARGET_FPS))
    cap.release()

    steps = [fis[i+1] - fis[i] for i in range(len(fis)-1)]
    wrong = [s for s in steps if s != fstp]
    if wrong:
        failures.append(
            f"ep {ep_id} (fstp={fstp}): {len(wrong)}/{len(steps)} steps != {fstp}: e.g. {wrong[:5]}"
        )

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} sampling-pattern failures")

print(f"PASS Test 3 — all episodes have monotonically increasing frame_index with step == fstp.")

---
## Test 4 — State Alignment (cross-check vs. base parquet)

**Guarantees:** For `N_SPOT_CHECK` randomly chosen episodes, the `states` vector at every frame
exactly matches `parquet['observation.state'][frame_index][STATE_START:STATE_START+STATE_DIM]`.
Rules out off-by-one index errors, wrong state slice, and wrong episode→parquet mapping.

**Does NOT guarantee:** Correctness for episodes not in the spot-check sample.

In [ ]:
rng = np.random.default_rng(42)
check_ids = rng.choice(list(episodes.keys()), size=min(N_SPOT_CHECK, len(episodes)), replace=False)

state_failures = []
for ep_id in check_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["frame_pos"])
    pp = parquet_path(sorted_rows[0]["sample_idx"])
    if not pp.exists():
        print(f"  SKIP ep {ep_id}: parquet not found")
        continue

    df = pd.read_parquet(pp, columns=["observation.state"])
    raw = np.asarray(df["observation.state"].to_list(), dtype=np.float32)

    for row in sorted_rows:
        fi = row["frame_index"]
        if fi >= len(raw):
            state_failures.append(f"ep {ep_id} frame_index={fi} >= parquet len {len(raw)}")
            continue
        expected = raw[fi, STATE_START : STATE_START + STATE_DIM]
        if not np.allclose(expected, row["states"], rtol=1e-4, atol=1e-5):
            state_failures.append(
                f"ep {ep_id} frame_pos={row['frame_pos']} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['states'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} frames checked — OK")

if state_failures:
    for f in state_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(state_failures)} state alignment failures")

print("PASS Test 4 — states == parquet observation.state[frame_index, 0:7] for all spot-checked episodes.")

---
## Test 5 — Action Aggregation (cross-check vs. base parquet)

**Guarantees:** For `N_SPOT_CHECK` episodes, `actions` at frame `k` equals
`flatten(raw_actions[k : next_k, :ACTION_DIM])` padded with repetition of the last action to
length `fstp`, matching exactly the logic in `behavior.py:loadvideo_decord`.  Verifies:
- Correct raw-to-sampled frame index mapping (no off-by-one).
- Correct aggregation window `[frame_k, frame_{k+1})`.
- Correct padding when near the episode boundary.

**Does NOT guarantee:** Correctness for episodes outside the spot-check.

In [ ]:
action_failures = []
for ep_id in check_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["frame_pos"])
    sid = sorted_rows[0]["sample_idx"]
    vp, pp = video_path(sid), parquet_path(sid)

    if not vp.exists() or not pp.exists():
        print(f"  SKIP ep {ep_id}: video or parquet not found")
        continue

    cap = cv2.VideoCapture(str(vp))
    vfps = cap.get(cv2.CAP_PROP_FPS)
    n_vid = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    fstp = max(1, ceil(vfps / TARGET_FPS))

    df = pd.read_parquet(pp, columns=["action"])
    raw_act = np.asarray(df["action"].to_list(), dtype=np.float32)[:, :ACTION_DIM]
    max_len = min(n_vid, len(raw_act))
    sampled = np.arange(0, max_len, fstp, dtype=np.int64)  # same as _episode_sampled_indices

    for row in sorted_rows:
        fi  = row["frame_index"]
        pos = row["frame_pos"]

        # Reconstruct the action window boundaries
        next_fi = int(sampled[pos + 1]) if pos + 1 < len(sampled) else fi + fstp
        end = min(max(next_fi, fi + 1), max_len)
        chunk = raw_act[fi:end]

        if len(chunk) == 0:
            chunk = np.zeros((fstp, ACTION_DIM), dtype=np.float32)
        elif len(chunk) < fstp:
            chunk = np.concatenate([chunk, np.repeat(chunk[-1:], fstp - len(chunk), axis=0)])
        else:
            chunk = chunk[:fstp]

        expected = chunk.reshape(fstp * ACTION_DIM)
        if not np.allclose(expected, row["actions"], rtol=1e-3, atol=1e-4):
            action_failures.append(
                f"ep {ep_id} frame_pos={pos} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['actions'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} action chunks checked — OK")

if action_failures:
    for f in action_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(action_failures)} action aggregation failures")

print("PASS Test 5 — actions == flatten(raw_actions[frame_index:next_frame, :23]) for all spot-checked episodes.")

---
## Test 6 — Token Shape Consistency & Finiteness

**Guarantees:** Every row has the *same* `(tokens_per_frame, hidden_dim)` shape (no mixed-shard
accidents) and all values are finite (no NaN / Inf from fp16 overflow or encoder crash).

**Does NOT guarantee:** Semantic quality of embeddings, or that bfloat16→float16 conversion did
not cause silent precision loss.

In [ ]:
shape_failures, nan_rows = [], []
for i, row in enumerate(rows):
    if row["tokens"].shape != TOKEN_SHAPE:
        shape_failures.append(f"row {i}: expected {TOKEN_SHAPE}, got {row['tokens'].shape}")
    if not np.all(np.isfinite(row["tokens"])):
        nan_rows.append(i)

if shape_failures:
    for f in shape_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(shape_failures)} token shape mismatches")

assert TOKEN_SHAPE[0] > 0 and TOKEN_SHAPE[1] > 0, "Degenerate token shape"

print(f"Token shape: {TOKEN_SHAPE}")
if nan_rows:
    print(f"WARNING — {len(nan_rows)} / {len(rows)} rows contain NaN or Inf in tokens")
else:
    print(f"PASS Test 6 — all {len(rows):,} token arrays have shape {TOKEN_SHAPE} and are finite.")

---
## Test 7 — Episode Coverage vs. Manifest

**Guarantees:** Every episode in the manifest appears in the MDS (no silent skip), and
no extra `sample_idx` values exist that have no corresponding manifest entry.

**Does NOT guarantee:** That the *frames* within each episode are correct (Tests 2–5 cover that).

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
n_manifest = len(manifest["episodes"])
manifest_indices = set(range(n_manifest))

mds_sample_indices = {row["sample_idx"] for row in rows}

missing = manifest_indices - mds_sample_indices
extra   = mds_sample_indices - manifest_indices

if missing:
    for idx in sorted(missing):
        ep = manifest["episodes"][idx]
        print(f"FAIL: sample_idx={idx} ({ep['task_name']}) missing from MDS")
    raise AssertionError(f"{len(missing)} manifest episodes missing from MDS")

if extra:
    raise AssertionError(f"MDS contains sample_idx values not in manifest: {sorted(extra)}")

print(f"PASS Test 7 — all {n_manifest} manifest episodes present in MDS, no extras.")

---
## Test 8 — Episode Frame Count vs. Expected Sampled Count

Replaces the "window boundaries" gap.  The window/fpc structure is dissolved in the MDS
(rows are per-frame), so window *grouping* cannot be directly inspected.  The meaningful
residual question is: did the preencoder store the right *number* of frames per episode?

**Guarantees:** For every episode, `episode_len` in the MDS equals
`len(np.arange(0, min(n_video_frames, parquet_len), fstp))` — i.e., exactly the frame count
that `BehaviorVideoDataset._episode_sampled_indices` would produce.  Rules out an entire
window being silently dropped or duplicated during the accumulate/flush phase.

**Does NOT guarantee:** That the pixel content of each frame is correct.

In [ ]:
from math import ceil

failures = []
for ep_id, ep_rows in episodes.items():
    stored_len = ep_rows[0]["episode_len"]
    sid = ep_rows[0]["sample_idx"]

    vp = video_path(sid)
    pp = parquet_path(sid)
    if not vp.exists() or not pp.exists():
        print(f"  SKIP ep {ep_id}: files not found locally")
        continue

    cap = cv2.VideoCapture(str(vp))
    vfps = cap.get(cv2.CAP_PROP_FPS)
    n_vid = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    fstp = max(1, ceil(vfps / TARGET_FPS))

    # Parquet length (same as during planning)
    parquet_len = len(pd.read_parquet(pp, columns=["action"]))
    max_len = min(n_vid, parquet_len)

    expected_len = len(np.arange(0, max_len, fstp, dtype=np.int64))

    if stored_len != expected_len:
        failures.append(
            f"ep {ep_id} (sample_idx={sid}): "
            f"stored episode_len={stored_len}, expected={expected_len} "
            f"(n_vid={n_vid}, parquet_len={parquet_len}, fstp={fstp})"
        )
    else:
        print(f"  ep {ep_id}: episode_len={stored_len} == expected {expected_len}  OK")

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} episode frame-count mismatches")

print("PASS Test 8 — all episode frame counts match expected sampled count from source video.")

---
## Notes on remaining gaps

**bfloat16 → float16 precision** — Not a separate test needed.  bf16 has fewer mantissa
bits (7) than fp16 (10), so converting bf16 → fp16 *increases* precision.  The only real
failure mode is range overflow (`|v| > 65504` → `inf`), which Test 6 already catches via
`np.isfinite`.

**Action aggregation at episode boundaries** — Already handled in Test 5.  The
reconstruction uses the same `max_len = min(n_vid, parquet_len)` and
`sampled = np.arange(0, max_len, fstp)`.  For window-final frames (where
`loadvideo_decord` uses `next_start = start + fstp`), the reconstructed
`sampled[pos+1] = fi + fstp` is algebraically identical because sampling is uniform.

**Pixel / encoder correctness** — Cannot be tested without re-running the encoder on the
raw video and comparing token values numerically.  This would require GPU + the full
V-JEPA2 model and is outside the scope of a lightweight dataset validation notebook.

---
## Final Summary

| Test | What it guarantees | Remaining gap |
|------|--------------------|---------------|
| 1 Schema | All MDS columns present, correct shapes | Not numerical values |
| 2 Completeness | No missing/duplicate frames per episode | Not whether episode count matches manifest |
| 3 Sampling | Correct 5 fps subsampling (step == fstp) | Not pixel content of decoded frames |
| 4 States | states == parquet[frame_index][0:7] | Only N_SPOT_CHECK episodes |
| 5 Actions | actions == aggregated chunk from parquet (incl. boundary) | Only N_SPOT_CHECK episodes |
| 6 Tokens | Consistent shape, all finite (incl. bf16→fp16 overflow) | Semantic quality of embeddings |
| 7 Coverage | Every manifest episode present in MDS, no extras | Frame-level correctness within episode |
| 8 Frame count | episode_len matches expected sampled count per source video | Pixel content; window ordering within episode |

**Fundamentally untestable here:** pixel correctness and encoder output quality — both
require re-running the full GPU inference pipeline.